# MemSum Human Evaluation

## Clone repos, Install Dependencies, Set working directory (takes about 4 minutes)



In [2]:
#@title Mount Drive, Set save and load folder (folder that contains the notebooks)
from google.colab import drive
import glob
import os

drive.mount('/content/drive')

gdrive_folder_name = ''
drive_folders = glob.glob('/content/drive/MyDrive/*')
for drive_folder in drive_folders:
  if drive_folder.startswith('/content/drive/MyDrive/2025_ReproNLP'):
    head, tail = os.path.split(drive_folder)
    gdrive_folder_name = tail

print(f'Annotation folder: {gdrive_folder_name}')

Mounted at /content/drive
Annotation folder: 2025_ReproNLP


In [3]:
#@title Clone MemSum, change working directory
!git clone https://github.com/nianlonggu/MemSum.git

import os
import json
os.chdir("MemSum")

#SM---START---
def prepare_json():
  human_eval_data_orig = [ json.loads(line) for line in open('human_eval_results/records_memsum_wo_autostop_neusum.jsonl',"r") ]
  human_eval_data_orig_no_rankings = []
  x = 0
  while x < len(human_eval_data_orig):
    new_dict = {}
    for key in human_eval_data_orig[x].keys():
      if key != 'ranking_results':
        new_dict[key] = human_eval_data_orig[x][key]
    human_eval_data_orig_no_rankings.append(new_dict)
    x += 1
  with open('human_eval_results/records_memsum_wo_autostop_neusum_clean.jsonl', 'w') as f:
    for item in human_eval_data_orig_no_rankings:
      f.write(json.dumps(item) + '\n')
  !rm /content/MemSum/human_eval_results/records_memsum_wo_autostop_neusum.jsonl
  !rm /content/MemSum/human_eval_results/records_memsum_neusum.jsonl

prepare_json()
#SM---END---

Cloning into 'MemSum'...
remote: Enumerating objects: 436, done.
remote: Counting objects: 100% (113/113), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 436 (delta 90), reused 75 (delta 74), pack-reused 323 (from 1)
Receiving objects: 100% (436/436), 82.74 MiB | 27.70 MiB/s, done.
Resolving deltas: 100% (182/182), done.


In [4]:
#@title Requirements
!pip install -r requirements.txt --quiet

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.5/60.5 kB 2.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 89.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.9/386.9 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.7/59.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 66.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.5/133.5 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 5.9 MB/s eta 0:00:00


In [5]:
#@title Clone Sent2Vec
!git clone https://github.com/epfml/sent2vec/
!cd sent2vec && git reset --hard 770bd2d && make && pip install .

Cloning into 'sent2vec'...
remote: Enumerating objects: 425, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 425 (delta 9), reused 4 (delta 1), pack-reused 403 (from 1)
Receiving objects: 100% (425/425), 447.46 KiB | 3.86 MiB/s, done.
Resolving deltas: 100% (261/261), done.
HEAD is now at 770bd2d Update README.md
c++ -pthread -std=c++0x -O3 -funroll-loops -c src/args.cc
c++ -pthread -std=c++0x -O3 -funroll-loops -c src/dictionary.cc
c++ -pthread -std=c++0x -O3 -funroll-loops -c src/productquantizer.cc
c++ -pthread -std=c++0x -O3 -funroll-loops -c src/matrix.cc
c++ -pthread -std=c++0x -O3 -funroll-loops -c src/shmem_matrix.cc
c++ -pthread -std=c++0x -O3 -funroll-loops -c src/qmatrix.cc
c++ -pthread -std=c++0x -O3 -funroll-loops -c src/vector.cc
c++ -pthread -std=c++0x -O3 -funroll-loops -c src/model.cc
c++ -pthread -std=c++0x -O3 -funroll-loops -c src/utils.cc
c++ -pthread -std=c++0x -O3 -funroll-loops -c src/fasttext.cc

In [6]:
#@title Change working directory
os.chdir("../MemSum")

In [ ]:
#@title Load Sent2Vec
import sent2vec

global sent2vec_model
sent2vec_model = sent2vec.Sent2vecModel()
sent2vec_model.load_model('/content/drive/MyDrive/'+gdrive_folder_name+'/wiki_unigrams.bin')

## Utils for evaluation interface

In [1]:
#@title Eval code
import requests
import json
import ipywidgets as widgets
from ipywidgets import Layout, Button, Box, FloatText, Textarea, Dropdown, Label, IntSlider, GridspecLayout
from IPython.display import display, Markdown, clear_output
import numpy as np
import pprint
import nltk
from scipy.stats import ttest_rel , ttest_ind, wilcoxon
from scipy.spatial.distance import cosine
import sys
import pickle

#SM---START---
from requests import get
filename = get('http://172.28.0.12:9000/api/sessions').json()[0]['name']
id_notebook = filename.rsplit('_', 1)[1].rsplit('.', 1)[0]

load_partially_completed_file = True #@param{type:"boolean"}
path_file_to_load = "/content/drive/MyDrive/"+gdrive_folder_name+"/saved_annotations"+id_notebook
# path_file_to_load = "/content/drive/MyDrive/2025_ReproNLP_Annotators_DCU/saved_annotations"+id_notebook
if load_partially_completed_file == True:
  if path_file_to_load == None:
    print('ERROR: Please enter a filepath.')
    sys.exit()
  else:
    if os.path.exists(path_file_to_load) == False:
      print('ERROR: Filepath does not exist.')
      sys.exit()
    else:
      print('Found file to load.')
#SM---END---

def get_summ_example():
    global human_eval_data, current_doc_idx, num_of_eval_docs

    found = False
    for pos in range( current_doc_idx, len(human_eval_data) ):
        #SM---START---
        # Commented this block, which was probably there for showing the results only (it skips evaluation items)
        # if human_eval_data[pos]["ranking_results"]["overall"] == [1,1]:
        #     human_eval_data[pos]["new_human_eval_results"] = human_eval_data[pos]["ranking_results"]
        #     num_of_eval_docs += 1
        #     if num_of_eval_docs >= len(human_eval_data):
        #         submit_button.disabled = True
        # else:
        #SM---END---
        #SM---START---
        # Unindented this block
        found = True
        current_doc_idx = pos
        break
        #SM---END---
    if found:
        summ_example = human_eval_data[current_doc_idx]
        current_doc_idx = min(current_doc_idx+1, len(human_eval_data) )
    else:
        summ_example = None
        current_doc_idx = len(human_eval_data)

    return summ_example


class TextHTML(widgets.HTML):
    def __init__(self, html_style = {} ,**kwargs):
        super().__init__(**kwargs )
        self.default_html_style = {
            "padding":"5px",
            "height":"600px",
            "overflow-x":"hidden",
            "border":"1px solid grey",
            "line-height":"20px"
         }
        self.render_sen_list(html_style=html_style)
        self.html_lines = []

    def render_sen_list(self, sens=[], html_style = {}):
        self.default_html_style.update(html_style)
        html_lines = [
            '''<div style="%s">''' %( "; ".join( ":".join([key, value]) for key, value in self.default_html_style.items() )  )
        ]

        for sen in sens:
            is_marked = sen.get("is_marked", False)
            sen_text = sen.get("text", "").capitalize()
            html_line = '''<p> %s %s %s</p>'''%( '''<span style="background-color: #FFFF00">''' if is_marked else "",
                                             sen_text,
                                             '''</span>''' if is_marked else ""
                                           )
            html_lines.append( html_line )

        html_lines.append( "</div>" )
        value = "\n".join(html_lines)
        self.value = value
        self.html_lines = html_lines

    def update_html_style( self, html_style = {} ):
        self.default_html_style.update(html_style)
        self.html_lines[0] = '''<div style="%s">''' %( "; ".join( ":".join([key, value]) for key, value in self.default_html_style.items() )  )
        value = "\n".join(self.html_lines)
        self.value = value


def get_cosine_sim(query, sentences, model):
  query_vec = model.embed_sentence(query)
  sentences_vec = model.embed_sentences(sentences)
  cosine_sim = [ 1-cosine(query_vec[0], sent_vec) for sent_vec in sentences_vec ]
  return cosine_sim


# Function to highlight text when the button is pressed
def on_highlight_button_click(change):
    # Update the HTML for each summary with highlighted text
    global summ_example, query_text, sent2vec_model
    query = query_text.value

    # TODO highlight query

    summaryA = summ_example["random_extracted_results"][0][0]
    summaryA_cosine_sim = get_cosine_sim(query, summaryA, sent2vec_model)
    list_sent = []
    for idx, sent in enumerate(summaryA):
      if summaryA_cosine_sim[idx]>0.6:
        list_sent.append({"text":f'<mark>{sent}</mark>'})
      else:
        list_sent.append({"text":sent})
    text_summ_sources["Summary A"].render_sen_list( list_sent )

    summaryB = summ_example["random_extracted_results"][1][0]
    summaryB_cosine_sim = get_cosine_sim(query, summaryB, sent2vec_model)
    list_sent = []
    for idx, sent in enumerate(summaryB):
      if summaryB_cosine_sim[idx]>0.6:
        list_sent.append({"text":f'<mark>{sent}</mark>'})
      else:
        list_sent.append({"text":sent})
    text_summ_sources["Summary B"].render_sen_list( list_sent )


form_item_layout = Layout(
    display='flex',
    flex_flow='row',
    justify_content='space-between'
)

rb_criteria = {}
for criterion in ["Overall:"]:
    rb_criteria[criterion] =  widgets.RadioButtons(
                options=['summary A', 'summary B'],
                disabled=False,
                index = None
    )

b_summ_sources = {}
colors_for_b_summ_sources ={ "Reference Summary":"YellowGreen","Summary A":"lightblue", "Summary B":"lightblue" }
for source in ["Reference Summary", "Summary A", "Summary B"]:
    b_summ_sources[source] = Button(description=source, layout=Layout(height='auto', width='auto'))
    b_summ_sources[source].style.button_color = colors_for_b_summ_sources[source]

text_summ_sources = {}
for source in ["Reference Summary", "Summary A", "Summary B"]:
    text_summ_sources[source] =  TextHTML({"height":"500px"})  # Textarea(layout=Layout(height="600px", width='auto'))

submit_button = Button(description="Submit & Eval Next", layout=Layout(height='auto', width='auto'))
submit_button.style.button_color = "LightSalmon"


global fulltext_textbox
fulltext_button = Button(description="Show Source Document >>>", layout=Layout(height='auto', width='32.9%'))
fulltext_textbox = TextHTML( html_style={"height":"0px"}, layout=Layout(visibility="hidden") )

grid_b_summ = GridspecLayout(1,3)
grid_b_summ[0,0] = b_summ_sources["Reference Summary"]
grid_b_summ[0,1] = b_summ_sources["Summary A"]
grid_b_summ[0,2] = b_summ_sources["Summary B"]
grid_text_summ = GridspecLayout(1,3)
grid_text_summ[0,0] = text_summ_sources["Reference Summary"]
grid_text_summ[0,1] = text_summ_sources["Summary A"]
grid_text_summ[0,2] = text_summ_sources["Summary B"]
grid_rb_description = GridspecLayout(1,3)
grid_rb_description[0,0] = Label(value = "Overall:")


global query_text
highlight_button = Button(description="Highlight relevant sentences given a query 🔍", layout=Layout(height='auto', width='auto'))
highlight_button.on_click(on_highlight_button_click)
query_text = Textarea(placeholder='Enter your query here...', layout=Layout(width='auto', height='auto'))

grid_query = GridspecLayout(1,3)
grid_query[0,0] = highlight_button
grid_query[0,1:] = query_text

output_panel = widgets.Output()


form_items = [
    widgets.HTML(value = f"<b><font color='black' font size='4pt'>Read</b>"),
    grid_query,
    grid_b_summ,
    grid_text_summ,
    fulltext_button,
    fulltext_textbox,
    widgets.HTML(value = f"<b><font color='black' font size='4pt'>Evaluation (choose one that is closer to the reference summary)</b>"),
    grid_rb_description,
    widgets.HBox([ rb_criteria["Overall:"] ], layout=form_item_layout),
    widgets.Box([  submit_button ], layout=form_item_layout),
    output_panel

]

gui = Box(form_items, layout=Layout(
    display='flex',
    flex_flow='column',
    border='solid 2px',
    align_items='stretch',
    width='100%'
))


def get_next_example():
    global summ_example, num_of_eval_docs, human_eval_data
    summ_example = get_summ_example()
    if summ_example is not None:
        text_summ_sources["Reference Summary"].render_sen_list( [{"text":_} for _ in summ_example["summary"]] )
        text_summ_sources["Summary A"].render_sen_list( [{"text":_} for _ in summ_example["random_extracted_results"][0][0] ] )
        text_summ_sources["Summary B"].render_sen_list( [{"text":_} for _ in summ_example["random_extracted_results"][1][0] ] )
    else:
        text_summ_sources["Reference Summary"].render_sen_list( [] )
        text_summ_sources["Summary A"].render_sen_list( [] )
        text_summ_sources["Summary B"].render_sen_list( [] )


    for criterion in ["Overall:"]:
        rb_criteria[criterion].index = None

    if summ_example is not None:
        fulltext_textbox.render_sen_list( [{"text":_} for _ in summ_example["text"]] )
    else:
        fulltext_textbox.render_sen_list( [] )

    fulltext_textbox.update_html_style({"height":"0px"})
    fulltext_textbox.layout.visibility = "hidden"
    fulltext_button.description = "Show Source Document >>>"

    # empty search textbox
    query_text.value = ""

def fulltext_button_on_click_listener(_):
    if fulltext_button.description == "Show Source Document >>>":
        fulltext_textbox.update_html_style({"height":"600px"})
        fulltext_textbox.layout.visibility = "visible"
        fulltext_button.description = "Hide Source Document >>>"
    elif fulltext_button.description == "Hide Source Document >>>":
        fulltext_textbox.update_html_style({"height":"0px"})
        fulltext_textbox.layout.visibility = "hidden"
        fulltext_button.description = "Show Source Document >>>"
fulltext_button.on_click( fulltext_button_on_click_listener )


def submit_button_on_click_listener(_):
    global summ_example, num_of_eval_docs
    all_evaluated = True
    for criterion in ["Overall:"]:
        if rb_criteria[criterion].index is None:
            with output_panel:
                clear_output()
                print("You have not evaluated %s, please retry."%( criterion.rstrip(":") ))
            all_evaluated = False
    if all_evaluated:
        two_orders = [ [1,2],[2,1] ]
        summ_example["new_human_eval_results"] = {
                                         "overall":two_orders[ rb_criteria["Overall:"].index ]
                                    }
        num_of_eval_docs += 1
        if num_of_eval_docs >= len(human_eval_data):
            submit_button.disabled = True
        else:
            get_next_example()
        with output_panel:
            clear_output()
            print("You have evaluated %d/%d examples."%( num_of_eval_docs, len(human_eval_data)))

    #SM---START---
    # print(num_of_eval_docs, current_doc_idx)
    # print(human_eval_data[num_of_eval_docs-1]["new_human_eval_results"])
    with open(path_file_to_load, 'wb') as f:
        pickle.dump(human_eval_data, f)
    #SM---END---

submit_button.on_click( submit_button_on_click_listener )

human_eval_data = None
summ_example = None
num_of_eval_docs = 0
current_doc_idx = 0

def run_gui( dataset_path,  width ="90%", textbox_height = "400px", ):
    global human_eval_data, summ_example, num_of_eval_docs, current_doc_idx
    #SM---START---
    num_annnotations_loaded = 0
    if load_partially_completed_file == True:
        if load_partially_completed_file == True:
            with open(path_file_to_load, 'rb') as f:
                human_eval_data = pickle.load(f)
                for z in range(len(human_eval_data)):
                    if 'new_human_eval_results' in human_eval_data[z].keys():
                        num_annnotations_loaded += 1
    else:
    #SM---END---
        human_eval_data = [ json.loads(line) for line in open(dataset_path,"r") ]

    summ_example = None
    num_of_eval_docs = 0
    current_doc_idx = 0
    #SM---START---
    if load_partially_completed_file == True:
        num_of_eval_docs = num_annnotations_loaded
        current_doc_idx = num_annnotations_loaded
    #SM---END---
    get_next_example()
    with output_panel:
        clear_output()
        print("You have evaluated %d/%d examples."%( num_of_eval_docs, len(human_eval_data)))

    for source in ["Reference Summary", "Summary A", "Summary B"]:
        text_summ_sources[source].update_html_style({"height":textbox_height})
    gui.layout.width = width
    return gui

NameError: name 'gdrive_folder_name' is not defined

## Human Evaluation Experiment II:

In [ ]:
run_gui(dataset_path = "human_eval_results/records_memsum_wo_autostop_neusum_clean.jsonl")